# Modélisation XGBoost — scénario météo parfaite


## Prochaine étape

Entraînement du modèle XGBoost sur `train`, évaluation sur `val` (globalement et par `horizon_h`), comparaison aux baselines.

L'erreur suit une forme en "W" :

- Horizons 0-5 (minuit-5h) : MAE la plus basse (~730-790 MW). Nuit = consommation stable et très prévisible.

- Horizons 6-9 (lever du jour, pointe du matin) : ça grimpe fort (jusqu'à 1115 MW à h8). Le réveil/démarrage de l'activité économique est le moment le plus variable de la journée , logique que ce soit le plus dur à prévoir.

- Horizons 10-14 (plateau de journée) : ça redescend (~1000-1050 MW). Plateau stable, comme dans l' EDA.

- Horizons 15-19 (fin d'après-midi → pointe du soir) : le pic d'erreur du modèle (jusqu'à 1227 MW à h18), exactement le moment de la pointe du soir que l' EDA identifiait comme le pic de conso de la journée. C'est le moment le plus dur à prévoir car le plus variable (dépend fortement du froid, des habitudes).

- Horizons 20-23 : ça redescend progressivement vers la stabilité nocturne.

C'est un signal de qualité, pas un problème : le modèle a plus de mal exactement là où la vraie consommation est elle-même la plus volatile (les rampes du matin et du soir)

## test


In [16]:
from models.xgboost.entrainer_xgboost import *
# Chargement (si pas déjà fait)
train, val, test = charger_splits()
train, val, test = encoder_categorielles(train, val, test)

X_train, y_train, colonnes_features = separer_x_y(train)
X_val, y_val, _ = separer_x_y(val, colonnes_features)
X_test, y_test, _ = separer_x_y(test, colonnes_features)

# Si le modèle est déjà entraîné en mémoire (variable `modele`), pas besoin de
# ré-entraîner -- sinon :
# modele = entrainer_xgboost(X_train, y_train, X_val, y_val)

# Évaluation sur TEST (jamais vu, ni entraînement ni early stopping)
resultats_test, pred_test = evaluer(modele, X_test, y_test, horizon_h=test["horizon_h"])

print(f"MAE  : {resultats_test['mae']:.1f} MW")
print(f"RMSE : {resultats_test['rmse']:.1f} MW")
print(f"MAPE : {resultats_test['mape']:.2f} %")

print("\n--- Comparaison train / val / test ---")
resultats_train, _ = evaluer(modele, X_train, y_train)
resultats_val, _ = evaluer(modele, X_val, y_val)
print(f"Train : MAE {resultats_train['mae']:.1f}")
print(f"Val   : MAE {resultats_val['mae']:.1f}")
print(f"Test  : MAE {resultats_test['mae']:.1f}")

MAE  : 957.0 MW
RMSE : 1278.0 MW
MAPE : 1.88 %

--- Comparaison train / val / test ---
Train : MAE 521.7
Val   : MAE 933.3
Test  : MAE 957.0


Le modèle sur-estime légèrement certains pics (ex: le 6-7 janvier, la prévision dépasse nettement le réel — pic rouge à ~76 000 contre réel à ~68 000 MW). C'est cohérent avec ce qu'on avait vu dans la courbe par horizon : les heures de pointe (15h-19h) sont les plus dures à prédire, logique vu que ce sont les moments où le comportement humain (chauffage, activité) introduit le plus de variabilité non capturée par les lags.

## Analyse des échecs

In [24]:
import numpy as np
import pandas as pd

jours_a_comparer = ["2025-01-04", "2026-01-11", "2025-01-26", "2025-01-21", "2026-01-07",
                    "2025-11-17", "2026-01-12", "2026-02-15", "2026-01-13", "2025-01-24"]

# MAE de ces jours dans le scénario opérationnel (déjà obtenues)
mae_realiste = pd.Series({
    "2025-01-04": 8505.8, "2026-01-11": 7418.0, "2025-01-26": 7391.7, "2025-01-21": 7377.8,
    "2026-01-07": 6252.2, "2025-11-17": 6031.3, "2026-01-12": 5826.4, "2026-02-15": 4868.9,
    "2026-01-13": 4472.9, "2025-01-24": 4210.4,
})

diag_p = test[["date_prevision", "horizon_h", "cible_consommation_mw"]].copy()
diag_p["pred"] = modele.predict(X_test)      # modele = modèle météo parfaite de ce notebook
diag_p["err_abs"] = (diag_p["cible_consommation_mw"] - diag_p["pred"]).abs()
diag_p["err_signee"] = diag_p["pred"] - diag_p["cible_consommation_mw"]
diag_p["jour_J"] = diag_p["date_prevision"].astype(str).str[:10]

par_jour_p = diag_p.groupby("jour_J").agg(
    mae_parfait=("err_abs", "mean"),
    biais_parfait=("err_signee", "mean"),
)

comparaison = par_jour_p.loc[jours_a_comparer].copy()
comparaison["mae_realiste"] = mae_realiste
comparaison["reduction_%"] = (1 - comparaison["mae_parfait"] / comparaison["mae_realiste"]) * 100
display(comparaison.round(1))

print(f"MAE moyenne de ces 10 jours : réaliste {comparaison['mae_realiste'].mean():.0f} MW"
      f" -> parfaite {comparaison['mae_parfait'].mean():.0f} MW")

,mae_parfait,biais_parfait,mae_realiste,reduction_%
jour_J,,,,
2025-01-04,1197.5,957.0,8505.8,85.9
2026-01-11,2218.6,2218.6,7418.0,70.1
2025-01-26,2491.7,2491.7,7391.7,66.3
2025-01-21,2555.6,2555.6,7377.8,65.4
2026-01-07,694.5,233.8,6252.2,88.9
2025-11-17,2096.5,-2058.4,6031.3,65.2
2026-01-12,3300.7,3300.7,5826.4,43.3
2026-02-15,1823.3,1823.3,4868.9,62.6
2026-01-13,3106.3,3106.3,4472.9,30.6


MAE moyenne de ces 10 jours : réaliste 6236 MW -> parfaite 2140 MW


Analyse des échecs : parfait vs réaliste. Sur les 10 pires jours du scénario réaliste (MAE moyenne ≈ 6 200 MW), le même modèle entraîné avec la météo parfaite descend à ≈ 2 140 MW (−30 % à −89 % selon les jours). Une grande part des échecs vient donc de la persistance de la météo observée à 14h, qui ne peut pas anticiper un changement de temps (ex. 04/01/2025 : 8 506 → 1 198 MW ; 07/01/2026 : 6 252 → 695 MW). Cependant, certains jours (12 et 13/01/2026) restent mal prédits même avec la météo parfaite (≈ 3 100–3 300 MW d'erreur). L'erreur y est biaisée dans un seul sens sur toutes les heures, ce qui indique une limite propre au modèle : peu d'exemples de froid extrême dans le train, et incapacité des arbres à extrapoler.
 Pistes : une vraie prévision météo de J+1 (prioritaire), et des variables mieux adaptées aux extrêmes (degrés sous 15 °C plus fins, température lissée sur 24 à 48h, effet de la vague de froid cumulée).